# Overarching plan 
Two main changes are applied to the original model

## TECHNOLOGIES (For Irina) 
1. Add two technologies to the excel [Already done by Francesco]
- Gas boiler
- Heat pump 

2. For each technology add: 
- STATIC PARAMETERS: 
    - Type: Link 
    - Sector: Residential 
    - bus0: electricity (for Heat pump) and gas (for Gas boiler)
    - bus1: Residential heating high insulation 
    - bus2: emissions  
    - bus3: [ONLY for gas boiler] Residential heating low insulation 

- DYNAMIC PARAMETERS
    - CAPEX: [TBD] -> find out the baseline CAPEX for heat pumps and gas boilers
    - Subsidy: [TBD][ONLY for heat pumps] -> set up baseline subsidy for heat pumps (so stopping at 2030) 
    - FOM: [TBD] -> find out for both 
    - VOM: [TBD] -> find out for both
    - Efficiency [TBD] -> find out for both 
    - Lifetime: [TBD] -> find out for both 

3. Add a link with four buses [Already done by Francesco]

4. We changed how investment_cost is calculated to be able to sweep subsidy levels 

## DEMAND (For Anna) 
1. Add two demand in the excel: 
- Residential heating high insulation 
- Residential heating low insulation 

2. To each demand add: 
- [column] 'Total_demand': [TBD] -> total demand of energy for heating purposes in the NL for 2020 

3. We changed how demand is propagated to be able to sweep K (maximum saturation of highly insulated houses)


# CHANGES:
This section is dedicated to show the changes from the original model.py to the one used for this project 

## Default input parameters


In [2]:
# Default parameters for testing

# input database folder
db_dir = 'input_databases/default'

# model years
years = [2020,2030,2040,2050]

# intra-year temporal resolution
t_res = 15 # hours

# discount rates
rt = 0.07 # technical discount rate
rs = 0.01 # social discount rate

# passenger cars conversion factor
passenger_cars_cf = 40*0.1  # km/h

# NEW: subsidy growth 
subsidy_growth = 0.0

# NEW: demand parameters (insulation parameters)
K  = 0.9    # saturation: max share of highly insulated houses
s0 = 0.1    # share of highly insulated houses at t = 0
r  = 0.05   # growth rate
R  = 0.3    # demand reduction for highly insulated houses
T0 = 2000   # reference year, so t = model_year - 2000


## How to project demand 
1. Instead of a start value, we add in the demand excel a Total_demand column 
2. if the demand has such column, it's start_value is calculated as: 
    - (1-s)*D [for Residential heating low insulation]
    - s * D * (1 - R) [for Residential heating high insulation]

3. s stands for share of highly insulated houses, R stands for reduction in demand for highly insulated houses, and D is Total_demand 

4. s is calculated through a logistic growth function: 
    - 𝑠(𝑡) = 𝐾 / 1 + ((𝐾 − 𝑠0) / 𝑠0 ⋅ e^(−𝑟⋅𝑡)) 

    where K is saturation parameter for max share of houses that could be highly insulated, s0 is share of highly insulated houses at time 0, r is the rate of growth, t is years (i.e. 0 at 2020, 10 at 2030, etc.)

5. We need to define t as a [model_years - 2000]
6. Set D, R, s0, K, r, and t before the model_run function



In [ ]:
# Changes to preprocessing_demand in preprocessing.py

# Set logistic growth function 
def insulation_share(y, K=K, s0=s0, r=r, t0=T0):
    t = y - t0
    return K / (1 + ((K - s0) / s0) * np.exp(-r * t))


# Project_demand remains unchanges: we define a function that uses drivers_prj to project a start_value from base_year to year y, 
# based on the formula we studied in the class:
def project_demand(y, driver, start_value, base_year=years[0], dc=1):
    prj = start_value * (drivers_prj[drivers_prj.driver==driver][y].values[0]/drivers_prj[drivers_prj.driver==driver][base_year].values[0])**dc
    return prj


# We use function to create a dataframe with demand projections called dmd_prj:
dmd_prj = pd.DataFrame(index=demands.index, columns=years)
for i in demands.index:
    values = []
    for y in years:
        start_value = demands.loc[i, "Start_value"]

        # if the demand has a Total_demand, derive the start value from it
        if "Total_demand" in demands.columns and pd.notna(demands.loc[i, "Total_demand"]):
            D = demands.loc[i, "Total_demand"]
            s = insulation_share(y)

            if i == "Residential heat low insulation":
                start_value = (1 - s) * D
            elif i == "Residential heat high insulation":
                start_value = s * D * (1 - R)

        values.append(project_demand(y, demands.loc[i, "Driver"], start_value,
                                     dc=demands.loc[i, "Exponent"]))
    dmd_prj.loc[i, :] = values

display(dmd_prj)


,2020,2030,2040,2050
Demand,,,,
residential_appliances,18000000.0,18647151.700478,19090595.991683,19367491.123793
passenger_cars,100000.0,103801.763988,107013.503334,109831.460697


## How to determine investment cost
To be able to sweep capex for, we add parameter 'subsidy_growth': 
- if there is a subsidy for such technology, then the investment cost is calculated is capex - subsidy 
- to overcome the problem that the baseline brings subsidies to 0 after 2030, for scenarios where the subsidy is zero AND the subsidy_growth is above 0, we take the subsidy of the previous year and increase it by subsidy_growth


In [ ]:
## Changing investment_cost in model.py

def investment_cost(technology, year, techs_d, rt, subsidy_growth):
    capex = techs_d.at[(technology, 'CAPEX'), year]

    # only apply the subsidy mechanism if this technology has a Subsidy row
    if (technology, 'Subsidy') in techs_d.index: 
        subsidy = techs_d.at[(technology, 'Subsidy'), year]

        # if no subsidy this year and subsidy growth is higher than 0 [so not baseline scenario], take the previous year's subsidy
        if subsidy == 0 and subsidy_growth > 0.0:
            years = list(techs_d.columns)
            i = years.index(year)
            if i > 0:
                subsidy = techs_d.at[(technology, 'Subsidy'), years[i - 1]]

        subsidy = subsidy * (1 + subsidy_growth)
        capex = max(capex - subsidy, 0) # a subsidy shouldn't make CAPEX negative

    fom = techs_d.at[(technology,'FOM'),year] 
    lt = techs_d.at[(technology,'Lifetime'),year]
    return (capex*annuity(rt, lt) + fom) # (€/activity output unit) 
